In [10]:
import pandas as pd
import numpy as np

from pathlib import Path

In [11]:
data_path = Path.cwd().parent / ".data" / "flights_with_weather.parquet"
data_path

PosixPath('/home/beck/code/LuisHBeck/flight-risk/.data/flights_with_weather.parquet')

In [12]:
data = pd.read_parquet(data_path)
data.head(3)

,airline_icao,origin_icao,destination_icao,dep_scheduled,dep_actual,arr_scheduled,arr_actual,origin_type,origin_lat,origin_lon,...,origin_wx_cloudcover,origin_wx_weathercode,origin_wx_surface_pressure,destination_wx_temperature_2m,destination_wx_precipitation,destination_wx_windspeed_10m,destination_wx_windgusts_10m,destination_wx_cloudcover,destination_wx_weathercode,destination_wx_surface_pressure
0,TAM,SBRJ,SBGR,2022-01-06 14:20:00,2022-01-06 14:31:00,2022-01-06 15:25:00,2022-01-06 15:29:00,large_airport,-22.910397,-43.16282,...,92.0,3.0,1005.459106,22.6,5.4,17.873556,34.560001,98.0,63.0,926.656494
1,TAM,SBRJ,SBGR,2022-01-07 14:20:00,2022-01-07 14:47:00,2022-01-07 15:25:00,2022-01-07 15:42:00,large_airport,-22.910397,-43.16282,...,100.0,53.0,1009.652283,21.4,1.5,20.617662,39.599998,100.0,61.0,929.092102
2,TAM,SBRJ,SBGR,2022-01-08 14:20:00,2022-01-08 14:12:00,2022-01-08 15:25:00,2022-01-08 15:12:00,large_airport,-22.910397,-43.16282,...,100.0,53.0,1011.151184,20.0,0.2,22.907431,44.279999,100.0,51.0,929.817139


In [13]:
datetime_cols = [
    "dep_scheduled",
    "dep_actual",
    "arr_scheduled",
    "arr_actual"
]
data[datetime_cols] = (
    data[datetime_cols].apply(
        lambda col: pd.to_datetime(col, format="ISO8601") \
        .dt.floor("s") \
        .astype("datetime64[ns]")
    )
)

In [14]:
data.dtypes

airline_icao                                  str
origin_icao                                   str
destination_icao                              str
dep_scheduled                      datetime64[ns]
dep_actual                         datetime64[ns]
arr_scheduled                      datetime64[ns]
arr_actual                         datetime64[ns]
origin_type                                   str
origin_lat                                float64
origin_lon                                float64
origin_elevation_ft                       float64
origin_region                                 str
destination_type                              str
destination_lat                           float64
destination_lon                           float64
destination_elevation_ft                  float64
destination_region                            str
origin_wx_temperature_2m                  float32
origin_wx_precipitation                   float32
origin_wx_windspeed_10m                   float32


#### Target (delay)

In [15]:
# Atraso na partida e chegada em minutos
data['dep_delay_min'] = (data['dep_actual'] - data['dep_scheduled']).dt.total_seconds() / 60
data['arr_delay_min'] = (data['arr_actual'] - data['arr_scheduled']).dt.total_seconds() / 60

# Targets de classificação (critério ANAC: 15 min)
data['dep_is_delayed'] = (data['dep_delay_min'] > 15).astype(int)
data['arr_is_delayed'] = (data['arr_delay_min'] > 15).astype(int)

# Tempo ganho/perdido no ar (propagação do atraso)
data['delay_propagation'] = data['arr_delay_min'] - data['dep_delay_min']

data[['dep_delay_min', 'arr_delay_min', 'dep_is_delayed', 'arr_is_delayed', 'delay_propagation']].describe()

,dep_delay_min,arr_delay_min,dep_is_delayed,arr_is_delayed,delay_propagation
count,2.967376e+06,2.967376e+06,2.967376e+06,2.967376e+06,2.967376e+06
mean,4.400734e+00,1.845762e+00,1.494893e-01,1.528701e-01,-2.554971e+00
std,3.078920e+02,3.080145e+02,3.565702e-01,3.598623e-01,8.600936e+00
min,-5.249060e+05,-5.249070e+05,0.000000e+00,0.000000e+00,-1.230000e+03
25%,-7.000000e+00,-1.100000e+01,0.000000e+00,0.000000e+00,-8.000000e+00
50%,-2.000000e+00,-4.000000e+00,0.000000e+00,0.000000e+00,-3.000000e+00
75%,7.000000e+00,7.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00
max,4.463500e+04,4.462500e+04,1.000000e+00,1.000000e+00,2.020000e+02


#### Temporal

In [16]:
# Extrações básicas da partida programada
data['dep_hour']         = data['dep_scheduled'].dt.hour
data['dep_minute']       = data['dep_scheduled'].dt.minute
data['dep_day_of_week']  = data['dep_scheduled'].dt.dayofweek   # 0=seg, 6=dom
data['dep_month']        = data['dep_scheduled'].dt.month
data['dep_day_of_year']  = data['dep_scheduled'].dt.dayofyear
data['dep_is_weekend']   = (data['dep_day_of_week'] >= 5).astype(int)

# Hora cíclica (sen/cos para o modelo entender que 23h e 0h são próximos)
data['dep_hour_sin'] = np.sin(2 * np.pi * data['dep_hour'] / 24)
data['dep_hour_cos'] = np.cos(2 * np.pi * data['dep_hour'] / 24)

# Dia da semana cíclico
data['dep_dow_sin'] = np.sin(2 * np.pi * data['dep_day_of_week'] / 7)
data['dep_dow_cos'] = np.cos(2 * np.pi * data['dep_day_of_week'] / 7)

# Mês cíclico
data['dep_month_sin'] = np.sin(2 * np.pi * data['dep_month'] / 12)
data['dep_month_cos'] = np.cos(2 * np.pi * data['dep_month'] / 12)

data[['dep_hour', 'dep_day_of_week', 'dep_month', 'dep_is_weekend']].head()

,dep_hour,dep_day_of_week,dep_month,dep_is_weekend
0,14,3,1,0
1,14,4,1,0
2,14,5,1,1
3,14,6,1,1
4,14,1,1,0


In [17]:
# Time block
def time_block(hour):
    if 0 <= hour < 6:
        return "early_morning"
    if 6 <= hour < 12:
        return "morning"
    if 12 <= hour < 18:
        return "afternoon"
    return "evening"

data['dep_time_block'] = data['dep_hour'].apply(time_block)

# Horário de pico nos principais aeroportos brasileiros
data['dep_is_peak_hour'] = data['dep_hour'].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)

data['dep_time_block'].value_counts()

dep_time_block
afternoon        1009953
morning           975820
evening           710395
early_morning     271208
Name: count, dtype: int64

In [18]:
# Feriados nacionais brasileiros
import holidays

br_holidays = holidays.Brazil(years=range(2022, 2026))

data['dep_is_holiday'] = data['dep_scheduled'].dt.date.astype('datetime64[ns]').isin(
    pd.to_datetime(list(br_holidays.keys()))
).astype(int)

data['dep_is_holiday'].value_counts()

dep_is_holiday
0    2898937
1      68439
Name: count, dtype: int64

#### Route

In [19]:
# Rota como string e par de regiões
data['route']       = data['origin_icao'] + '_' + data['destination_icao']
data['region_pair'] = data['origin_region'] + '_' + data['destination_region']
data['same_region'] = (data['origin_region'] == data['destination_region']).astype(int)

data[['route', 'region_pair', 'same_region']].head()

,route,region_pair,same_region
0,SBRJ_SBGR,BR-RJ_BR-SP,0
1,SBRJ_SBGR,BR-RJ_BR-SP,0
2,SBRJ_SBGR,BR-RJ_BR-SP,0
3,SBRJ_SBGR,BR-RJ_BR-SP,0
4,SBRJ_SBGR,BR-RJ_BR-SP,0


In [20]:
# Distância haversine entre origem e destino (km)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

data['distance_km'] = haversine(
    data['origin_lat'], data['origin_lon'],
    data['destination_lat'], data['destination_lon']
)

# Voo curto / médio / longo
data['flight_range'] = pd.cut(
    data['distance_km'],
    bins=[0, 500, 1500, np.inf],
    labels=['short', 'medium', 'long']
)

data[['distance_km', 'flight_range']].describe()

,distance_km
count,2.967376e+06
mean,9.569972e+02
std,6.499061e+02
min,1.976669e+01
25%,4.217931e+02
50%,7.808894e+02
75%,1.329481e+03
max,5.239830e+03


In [21]:
# Diferença de elevação origem → destino
data['elevation_diff_ft'] = data['destination_elevation_ft'] - data['origin_elevation_ft']

# Encode ordinal do tipo de aeroporto
airport_size = {'small_airport': 1, 'medium_airport': 2, 'large_airport': 3}
data['origin_airport_size']      = data['origin_type'].map(airport_size)
data['destination_airport_size'] = data['destination_type'].map(airport_size)
data['both_large_airports']      = ((data['origin_airport_size'] == 3) & (data['destination_airport_size'] == 3)).astype(int)

data[['elevation_diff_ft', 'origin_airport_size', 'destination_airport_size']].head()

,elevation_diff_ft,origin_airport_size,destination_airport_size
0,2450.0,3.0,3.0
1,2450.0,3.0,3.0
2,2450.0,3.0,3.0
3,2450.0,3.0,3.0
4,2450.0,3.0,3.0


In [22]:
# Rotas troncais de alta frequência no Brasil
trunk_routes = {
    'SBRJ_SBGR', 'SBGR_SBRJ',  # Ponte Aérea RJ-SP
    'SBGR_SBBR', 'SBBR_SBGR',  # SP-Brasília
    'SBGR_SBSV', 'SBSV_SBGR',  # SP-Salvador
    'SBGR_SBRF', 'SBRF_SBGR',  # SP-Recife
    'SBGR_SBFZ', 'SBFZ_SBGR',  # SP-Fortaleza
    'SBGR_SBPA', 'SBPA_SBGR',  # SP-Porto Alegre
    'SBGR_SBBH', 'SBBH_SBGR',  # SP-BH
}
data['is_trunk_route'] = data['route'].isin(trunk_routes).astype(int)

data['is_trunk_route'].value_counts()

is_trunk_route
0    2777118
1     190258
Name: count, dtype: int64

#### Weather derivate 

In [23]:
# Classificação de condição climática por weathercode (Open-Meteo)
def wx_condition(code):
    if pd.isna(code):           return 'unknown'
    code = int(code)
    if code == 0:               return 'clear'
    if code in [1, 2, 3]:      return 'cloudy'
    if code in [45, 48]:       return 'fog'
    if code in range(51, 68):  return 'rain'
    if code in range(71, 78):  return 'snow'
    if code in range(80, 83):  return 'showers'
    if code in range(95, 100): return 'storm'
    return 'other'

data['origin_wx_condition']      = data['origin_wx_weathercode'].apply(wx_condition)
data['destination_wx_condition'] = data['destination_wx_weathercode'].apply(wx_condition)

# Flags binárias por tipo de condição
for prefix in ['origin', 'destination']:
    col = f'{prefix}_wx_condition'
    data[f'{prefix}_wx_is_fog']   = (data[col] == 'fog').astype(int)
    data[f'{prefix}_wx_is_rain']  = (data[col].isin(['rain', 'showers'])).astype(int)
    data[f'{prefix}_wx_is_storm'] = (data[col] == 'storm').astype(int)

data[['origin_wx_condition', 'destination_wx_condition']].value_counts().head(10)

origin_wx_condition  destination_wx_condition
cloudy               cloudy                      761566
clear                cloudy                      351109
cloudy               clear                       349491
                     rain                        338267
clear                clear                       337261
rain                 cloudy                      332797
                     rain                        226231
clear                rain                        137675
rain                 clear                       132977
snow                 cloudy                           1
Name: count, dtype: int64

In [24]:
data.columns

Index(['airline_icao', 'origin_icao', 'destination_icao', 'dep_scheduled',
       'dep_actual', 'arr_scheduled', 'arr_actual', 'origin_type',
       'origin_lat', 'origin_lon', 'origin_elevation_ft', 'origin_region',
       'destination_type', 'destination_lat', 'destination_lon',
       'destination_elevation_ft', 'destination_region',
       'origin_wx_temperature_2m', 'origin_wx_precipitation',
       'origin_wx_windspeed_10m', 'origin_wx_windgusts_10m',
       'origin_wx_cloudcover', 'origin_wx_weathercode',
       'origin_wx_surface_pressure', 'destination_wx_temperature_2m',
       'destination_wx_precipitation', 'destination_wx_windspeed_10m',
       'destination_wx_windgusts_10m', 'destination_wx_cloudcover',
       'destination_wx_weathercode', 'destination_wx_surface_pressure',
       'dep_delay_min', 'arr_delay_min', 'dep_is_delayed', 'arr_is_delayed',
       'delay_propagation', 'dep_hour', 'dep_minute', 'dep_day_of_week',
       'dep_month', 'dep_day_of_year', 'dep_is_week

### New columns explanation

# 📊 Documentação das Features — Flight Risk Dataset

## Visão Geral

O dataset original possuía **31 colunas** brutas provenientes da ANAC e da API de clima (Open-Meteo). Após o processo de feature engineering, o dataset passou a contar com **68 colunas**, adicionando **37 novas features** organizadas nas categorias abaixo.

---

## 🎯 Target / Atraso

Colunas derivadas dos horários programados e reais de partida e chegada.

| Coluna | Tipo | Descrição |
|---|---|---|
| `dep_delay_min` | float | Atraso na **partida** em minutos. Negativo = adiantado. Calculado como `dep_actual - dep_scheduled` |
| `arr_delay_min` | float | Atraso na **chegada** em minutos. Negativo = adiantado. Calculado como `arr_actual - arr_scheduled` |
| `dep_is_delayed` | int (0/1) | `1` se o atraso na partida foi superior a **15 minutos** (critério ANAC) |
| `arr_is_delayed` | int (0/1) | `1` se o atraso na chegada foi superior a **15 minutos** (critério ANAC) |
| `delay_propagation` | float | Diferença entre `arr_delay_min` e `dep_delay_min`. Positivo = voo perdeu tempo no ar; negativo = recuperou tempo |

---

## 🕐 Features Temporais

Extraídas de `dep_scheduled`. Encodings cíclicos (sin/cos) permitem que o modelo entenda a continuidade do tempo — por exemplo, que 23h e 0h são horários próximos.

| Coluna | Tipo | Descrição |
|---|---|---|
| `dep_hour` | int | Hora da partida programada (0–23) |
| `dep_minute` | int | Minuto da partida programada (0–59) |
| `dep_day_of_week` | int | Dia da semana (0 = segunda, 6 = domingo) |
| `dep_month` | int | Mês da partida (1–12) |
| `dep_day_of_year` | int | Dia do ano (1–366) |
| `dep_is_weekend` | int (0/1) | `1` se a partida ocorre no sábado ou domingo |
| `dep_hour_sin` | float | Componente seno da hora (encoding cíclico) |
| `dep_hour_cos` | float | Componente cosseno da hora (encoding cíclico) |
| `dep_dow_sin` | float | Componente seno do dia da semana (encoding cíclico) |
| `dep_dow_cos` | float | Componente cosseno do dia da semana (encoding cíclico) |
| `dep_month_sin` | float | Componente seno do mês (encoding cíclico) |
| `dep_month_cos` | float | Componente cosseno do mês (encoding cíclico) |
| `dep_time_block` | str | Bloco do dia: `madrugada` (0–5h), `manha` (6–11h), `tarde` (12–17h), `noite` (18–23h) |
| `dep_is_peak_hour` | int (0/1) | `1` se a partida está no horário de pico dos aeroportos (7–9h ou 17–20h) |
| `dep_is_holiday` | int (0/1) | `1` se a data de partida coincide com um feriado nacional brasileiro |

---

## ✈️ Features de Rota

Derivadas dos aeroportos de origem e destino e suas coordenadas geográficas.

| Coluna | Tipo | Descrição |
|---|---|---|
| `route` | str | Rota no formato `ORIGIN_DESTINATION` (ex: `SBRJ_SBGR`) |
| `region_pair` | str | Par de regiões no formato `BR-XX_BR-YY` (ex: `BR-RJ_BR-SP`) |
| `same_region` | int (0/1) | `1` se origem e destino estão na mesma região/estado |
| `distance_km` | float | Distância haversine em km entre as coordenadas de origem e destino |
| `flight_range` | str | Classificação do voo por distância: `curto` (< 500 km), `medio` (500–1500 km), `longo` (> 1500 km) |
| `elevation_diff_ft` | float | Diferença de altitude entre destino e origem em pés (`destination_elevation_ft - origin_elevation_ft`) |
| `origin_airport_size` | int | Tamanho do aeroporto de origem em escala ordinal: `small=1`, `medium=2`, `large=3` |
| `destination_airport_size` | int | Tamanho do aeroporto de destino em escala ordinal: `small=1`, `medium=2`, `large=3` |
| `both_large_airports` | int (0/1) | `1` se tanto origem quanto destino são aeroportos de grande porte |
| `is_trunk_route` | int (0/1) | `1` se a rota é considerada troncal de alta frequência no Brasil (ex: ponte aérea RJ-SP) |

---

## 🌦️ Features Meteorológicas Derivadas

Derivadas das colunas de clima brutas da origem e do destino (Open-Meteo).

| Coluna | Tipo | Descrição |
|---|---|---|
| `origin_wx_condition` | str | Condição climática na origem classificada pelo `weathercode`: `clear`, `cloudy`, `fog`, `rain`, `snow`, `showers`, `storm`, `other` |
| `destination_wx_condition` | str | Condição climática no destino (mesma classificação acima) |
| `origin_wx_is_fog` | int (0/1) | `1` se há neblina/nevoeiro na origem (weathercode 45 ou 48) |
| `origin_wx_is_rain` | int (0/1) | `1` se há chuva ou pancadas na origem (weathercodes 51–67 ou 80–82) |
| `origin_wx_is_storm` | int (0/1) | `1` se há tempestade na origem (weathercodes 95–99) |
| `destination_wx_is_fog` | int (0/1) | `1` se há neblina/nevoeiro no destino |
| `destination_wx_is_rain` | int (0/1) | `1` se há chuva ou pancadas no destino |
| `destination_wx_is_storm` | int (0/1) | `1` se há tempestade no destino |

---

## 📌 Resumo

| Categoria | Novas features |
|---|---|
| Target / Atraso | 5 |
| Temporais | 15 |
| Rota | 10 |
| Meteorológicas derivadas | 8 |
| **Total** | **38** |

### Model

In [25]:
def delay_class(minutes):
    if minutes <= 0:    return 0  # no horário
    if minutes <= 15:   return 1  # leve
    if minutes <= 60:   return 2  # moderado
    if minutes <= 120:  return 3  # severo
    return 4                      # crítico

data['delay_class'] = data['dep_delay_min'].apply(delay_class)

# Checar distribuição — IMPORTANTE fazer isso antes de decidir o modelo
print(data['delay_class'].value_counts(normalize=True).sort_index().mul(100).round(1))

delay_class
0    62.5
1    22.5
2    12.1
3     2.0
4     0.9
Name: proportion, dtype: float64


In [26]:
data = data.drop(columns=["delay_class"])

In [27]:
# melhor range para começar

# def delay_class_3(minutes):
#     if minutes <= 0:   return 0  # no horário
#     if minutes <= 60:  return 1  # aceitável
#     return 2                     # grave

def delay_class_3(minutes):
    if minutes <= 0:   return 0  # no horário
    if minutes <= 45:  return 1  # aceitável
    return 2

data['delay_class_3'] = data['dep_delay_min'].apply(delay_class_3)
print(data['delay_class_3'].value_counts(normalize=True).sort_index().mul(100).round(1))

delay_class_3
0    62.5
1    33.1
2     4.3
Name: proportion, dtype: float64


### Model training

In [28]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

In [29]:
# Target
TARGET = 'delay_class_3'

DROP_COLS = [
    # leakage
    'dep_scheduled', 'dep_actual', 'arr_actual', 'arr_scheduled',
    'dep_delay_min', 'arr_delay_min',
    'dep_is_delayed', 'arr_is_delayed',
    'delay_propagation',
    'arr_delay_min',
    # redundantes com encodings cíclicos
    'dep_hour', 'dep_minute', 'dep_day_of_week', 'dep_month',
]

feature_cols = [c for c in data.columns if c not in DROP_COLS + [TARGET]]

X = data[feature_cols].copy()
y = data[TARGET].copy()

print(f"Features: {X.shape[1]}")
print(f"Amostras: {X.shape[0]}")
print(f"\nDistribuição do target:\n{y.value_counts(normalize=True).sort_index().mul(100).round(1)}")

Features: 56
Amostras: 2967376

Distribuição do target:
delay_class_3
0    62.5
1    33.1
2     4.3
Name: proportion, dtype: float64


In [30]:
# encoder
CAT_COLS = [
    'airline_icao', 'origin_icao', 'destination_icao',
    'origin_type', 'destination_type',
    'origin_region', 'destination_region',
    'route', 'region_pair', 'flight_range',
    'dep_time_block',
    'origin_wx_condition', 'destination_wx_condition',
]

encoders = {}
for col in CAT_COLS:
    if col in X.columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le

print("Tipos restantes com object:")
print(X.dtypes[X.dtypes == 'object'])

Tipos restantes com object:
Series([], dtype: object)


In [31]:
# data split
data_sorted = data.sort_values('dep_scheduled') # prevent data leakage
X = X.loc[data_sorted.index]
y = y.loc[data_sorted.index]

# 70% treino | 15% validação | 15% teste
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, shuffle=False)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50, shuffle=False)

print(f"Treino:    {X_train.shape[0]:,}")
print(f"Validação: {X_val.shape[0]:,}")
print(f"Teste:     {X_test.shape[0]:,}")

Treino:    2,077,163
Validação: 445,106
Teste:     445,107


In [32]:
# model = lgb.LGBMClassifier(
#     objective='multiclass',
#     num_class=3,
#     class_weight='balanced',
#     n_estimators=1000,
#     learning_rate=0.05,
#     num_leaves=63,
#     min_child_samples=20,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42,
#     n_jobs=-1,
# )

# model.fit(
#     X_train, y_train,
#     eval_set=[(X_val, y_val)],
#     callbacks=[
#         lgb.early_stopping(stopping_rounds=50, verbose=True),
#         lgb.log_evaluation(period=100),
#     ]
# )

# print(f"\nMelhor iteração: {model.best_iteration_}")

In [33]:
# y_pred = model.predict(X_test)

# print("=" * 50)
# print("CLASSIFICATION REPORT")
# print("=" * 50)
# print(classification_report(
#     y_test, y_pred,
#     target_names=['No horário', 'Atraso aceitável', 'Atraso grave'],
#     digits=3
# ))

In [34]:
# cm = confusion_matrix(y_test, y_pred)
# disp = ConfusionMatrixDisplay(
#     confusion_matrix=cm,
#     display_labels=['No horário', 'Aceitável', 'Grave']
# )

# fig, ax = plt.subplots(figsize=(7, 6))
# disp.plot(ax=ax, cmap='Blues', colorbar=False)
# ax.set_title('Matriz de Confusão — LightGBM', fontsize=13)
# plt.tight_layout()
# plt.show()

In [35]:
# feat_imp = pd.DataFrame({
#     'feature': X.columns,
#     'importance': model.feature_importances_
# }).sort_values('importance', ascending=False).head(25)

# fig, ax = plt.subplots(figsize=(8, 8))
# sns.barplot(data=feat_imp, y='feature', x='importance', ax=ax, palette='viridis')
# ax.set_title('Top 25 Features — LightGBM', fontsize=13)
# ax.set_xlabel('Importance (gain)')
# plt.tight_layout()
# plt.show()

### Quick model boosting

In [36]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'num_leaves':        [31, 63, 127],
    'min_child_samples': [10, 20, 50],
    'learning_rate':     [0.01, 0.05, 0.1],
    'subsample':         [0.7, 0.8, 0.9],
    'colsample_bytree':  [0.7, 0.8, 0.9],
}

base_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    class_weight='balanced',
    n_estimators=300,  # menor pra rodar rápido
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

search = RandomizedSearchCV(
    base_model,
    param_distributions=param_grid,
    n_iter=20,               # 20 combinações aleatórias
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

search.fit(X_train, y_train)

print(f"Melhores parâmetros: {search.best_params_}")
print(f"Melhor f1_macro (cv): {search.best_score_:.3f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.7, learning_rate=0.01, min_child_samples=50, num_leaves=127, subsample=0.7; total time=196.2min
[CV] END colsample_bytree=0.7, learning_rate=0.01, min_child_samples=10, num_leaves=127, subsample=0.7; total time=196.9min
[CV] END colsample_bytree=0.7, learning_rate=0.01, min_child_samples=50, num_leaves=127, subsample=0.7; total time=198.2min
[CV] END colsample_bytree=0.7, learning_rate=0.01, min_child_samples=50, num_leaves=127, subsample=0.7; total time=198.3min
[CV] END colsample_bytree=0.8, learning_rate=0.1, min_child_samples=20, num_leaves=127, subsample=0.9; total time=156.8min
[CV] END colsample_bytree=0.8, learning_rate=0.1, min_child_samples=20, num_leaves=127, subsample=0.9; total time=163.3min
[CV] END colsample_bytree=0.7, learning_rate=0.01, min_child_samples=10, num_leaves=127, subsample=0.7; total time=188.5min
[CV] END colsample_bytree=0.7, learning_rate=0.01, min_child_samples=10, 

/home/beck/.pyenv/versions/3.12.9/envs/flight-risk/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END colsample_bytree=0.9, learning_rate=0.1, min_child_samples=50, num_leaves=63, subsample=0.8; total time=86.4min
[CV] END colsample_bytree=0.9, learning_rate=0.1, min_child_samples=50, num_leaves=63, subsample=0.8; total time=85.2min
[CV] END colsample_bytree=0.9, learning_rate=0.01, min_child_samples=20, num_leaves=63, subsample=0.8; total time=97.3min
[CV] END colsample_bytree=0.9, learning_rate=0.01, min_child_samples=20, num_leaves=63, subsample=0.8; total time=97.2min
[CV] END colsample_bytree=0.9, learning_rate=0.01, min_child_samples=20, num_leaves=63, subsample=0.8; total time=97.4min
[CV] END colsample_bytree=0.9, learning_rate=0.05, min_child_samples=50, num_leaves=127, subsample=0.9; total time=165.5min
[CV] END colsample_bytree=0.9, learning_rate=0.05, min_child_samples=50, num_leaves=127, subsample=0.9; total time=167.7min
[CV] END colsample_bytree=0.9, learning_rate=0.05, min_child_samples=50, num_leaves=127, subsample=0.9; total time=166.4min
[CV] END colsample_b

In [37]:
# Avaliar o melhor modelo no teste
y_pred_tuned = search.best_estimator_.predict(X_test)

print("=" * 50)
print("CLASSIFICATION REPORT — tuned")
print("=" * 50)
print(classification_report(
    y_test, y_pred_tuned,
    target_names=['No horário', 'Atraso aceitável', 'Atraso grave'],
    digits=3
))

CLASSIFICATION REPORT — tuned
                  precision    recall  f1-score   support

      No horário      0.760     0.521     0.618    272786
Atraso aceitável      0.458     0.550     0.500    152376
    Atraso grave      0.083     0.314     0.132     19945

        accuracy                          0.522    445107
       macro avg      0.434     0.462     0.417    445107
    weighted avg      0.626     0.522     0.556    445107

